# Colab Experiment Notebook

This notebook is the Colab version of your current thesis pipeline.

It is aligned with your actual setup:

- real images are prepared by you manually under `data/real_samples/`
- generator is fixed to `fairdiffusion`
- detectors are `CNNDetection + LGrad + NPR`
- `NPR` is assumed to already exist locally, so there is no separate download step here
- the formal experiment pipeline no longer exposes a fallback generator branch

Project root is fixed to:

- `/content/drive/MyDrive/project`


## 1. Mount Google Drive

Purpose:

- make the project directory available in Colab
- keep generated data and results persistent in Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


## 2. Central Config

Purpose:

- keep all important runtime parameters in one place
- avoid editing command lines everywhere

In [ ]:
PROJECT_ROOT = "/content/drive/MyDrive/project"
MODEL_ID = "runwayml/stable-diffusion-v1-5"
DEVICE = "cuda"
SAMPLES_PER_GROUP = 100
REAL_PER_GROUP = 100
SEED = 42

USE_CLIP = True
CLIP_MIN_SCORE = 0.18
GROUP_MARGIN_MIN = 0.01
HUMAN_PHOTO_MIN = 0.00
DISABLE_CARTOON_CHECK = True
DISABLE_TOY_CHECK = False
ALIGN_ON = "clip"

BOOTSTRAP_ITERS = 1000
DETECTORS = "cnndetection,lgrad,npr"


## 3. Enter Project and Check Structure

Purpose:

- verify the project directory exists
- verify required scripts and the local fairdiffusion package exist

In [ ]:
from pathlib import Path

project_root = Path(PROJECT_ROOT)
assert project_root.exists(), f"Project root not found: {PROJECT_ROOT}"

required_paths = [
    project_root / "requirements.txt",
    project_root / "scripts" / "01_generate.py",
    project_root / "scripts" / "02_quality_filter.py",
    project_root / "scripts" / "03_run_detectors.py",
    project_root / "scripts" / "04_fairness_eval.py",
    project_root / "scripts" / "05_gradcam_analysis.py",
    project_root / "scripts" / "06_consolidate_results.py",
    project_root / "scripts" / "06_patch_shuffling_exp.py",
    project_root / "scripts" / "07_physical_consistency.py",
    project_root / "scripts" / "08_high_pass_innovation.py",
    project_root / "scripts" / "09_master_report.py",
    project_root / "semantic-image-editing-main" / "semantic-image-editing-main",
]

missing = [str(p) for p in required_paths if not p.exists()]
assert not missing, "Missing required paths:\n" + "\n".join(missing)
print("Project structure check passed")


In [ ]:
%cd /content/drive/MyDrive/project


## 4. Install Dependencies

Purpose:

- install project dependencies
- install local `fairdiffusion` package via `requirements.txt`

Note:

- your `requirements.txt` now includes the local `semdiffusers` package

In [ ]:
!pip install -U pip
!pip install -r /content/drive/MyDrive/project/requirements.txt


In [ ]:
import importlib
for pkg in ["torch", "transformers", "diffusers", "accelerate", "semdiffusers"]:
    importlib.import_module(pkg)
print("Dependency import check passed")


## 5. Check GPU

Purpose:

- ensure Colab really has a usable GPU when `DEVICE = cuda`

In [ ]:
import torch
if DEVICE == "cuda":
    assert torch.cuda.is_available(), "CUDA requested but no GPU is available"
    print(torch.cuda.get_device_name(0))
else:
    print("Running in CPU mode")


## 6. Check Real Image Directories

Purpose:

- confirm your manually collected real images are already organized correctly
- this notebook does not use any crawler

Required directories:

- `data/real_samples/male-doctor`
- `data/real_samples/female-doctor`
- `data/real_samples/male-nurse`
- `data/real_samples/female-nurse`


In [ ]:
real_root = project_root / "data" / "real_samples"
groups = ["male-doctor", "female-doctor", "male-nurse", "female-nurse"]
image_exts = {".png", ".jpg", ".jpeg", ".webp", ".bmp"}

assert real_root.exists(), f"Real image root not found: {real_root}"
for group in groups:
    group_dir = real_root / group
    assert group_dir.exists(), f"Missing real image directory: {group_dir}"
    count = sum(1 for p in group_dir.iterdir() if p.is_file() and p.suffix.lower() in image_exts)
    print(group, count)


## 7. Check Detector Weights and Model Directories

Purpose:

- confirm `CNNDetection`, `LGrad`, `NPR` and preprocessing weights already exist
- there is no separate `NPR` download step in this notebook because you already have it

In [ ]:
external_root = project_root / ".external_models"
required_model_paths = [
    external_root / "weights" / "cnndetect" / "blur_jpg_prob0.5.pth",
    external_root / "weights" / "lgrad" / "LGrad-4class-Trainon-Progan_car_cat_chair_horse.pth",
    external_root / "weights" / "npr" / "NPR.pth",
    external_root / "weights" / "preprocessing" / "karras2019stylegan-bedrooms-256x256_discriminator.pth",
    external_root / "CNNDetection",
    external_root / "LGrad",
    external_root / "NPR-DeepfakeDetection",
]
missing_model_paths = [str(p) for p in required_model_paths if not p.exists()]
assert not missing_model_paths, "Missing model files or directories:\n" + "\n".join(missing_model_paths)

for rel in [
    ".external_models/weights/cnndetect",
    ".external_models/weights/lgrad",
    ".external_models/weights/npr",
    ".external_models/weights/preprocessing",
]:
    print("====", rel, "====")
    !ls -lh $PROJECT_ROOT/$rel


## 8. Stage 01 Generate Raw Metadata

Purpose:

- generate fake images with `fairdiffusion`
- register your local real images into metadata
- produce `data/metadata_raw.csv`


In [ ]:
generate_cmd = f"""
python /content/drive/MyDrive/project/scripts/01_generate.py \
  --project-root /content/drive/MyDrive/project \
  --real-source local \
  --generator fairdiffusion \
  --model-id {MODEL_ID} \
  --samples-per-group {SAMPLES_PER_GROUP} \
  --real-per-group {REAL_PER_GROUP} \
  --device {DEVICE} \
  --seed {SEED} \
  --overwrite
""".strip()
print(generate_cmd)
!{generate_cmd}


## 9. Check Stage 01 Output

Purpose:

- verify `metadata_raw.csv` exists
- inspect `group x y_true` counts
- this is your manual-real + fairdiffusion setup, not a crawler workflow

In [ ]:
import pandas as pd
metadata_raw_path = project_root / "data" / "metadata_raw.csv"
assert metadata_raw_path.exists(), f"Missing Stage 01 output: {metadata_raw_path}"
metadata_raw = pd.read_csv(metadata_raw_path)
assert len(metadata_raw) > 0, "metadata_raw.csv is empty"
print(metadata_raw.head())
print(metadata_raw.groupby(["group", "y_true"]).size())


## 10. Stage 02 Quality Filtering and Balancing

Purpose:

- compute quality score and CLIP-based semantic filtering
- filter both real and fake with the same logic
- balance by `group x y_true`
- produce `metadata_balanced.csv`


In [ ]:
quality_cmd = f"""
python /content/drive/MyDrive/project/scripts/02_quality_filter.py \
  --project-root /content/drive/MyDrive/project \
  --metadata-in /content/drive/MyDrive/project/data/metadata_raw.csv \
  --metadata-out /content/drive/MyDrive/project/data/metadata_balanced.csv \
  --balanced-dir /content/drive/MyDrive/project/data/generated_balanced \
  {'--use-clip' if USE_CLIP else ''} \
  --device {DEVICE} \
  --clip-min-score {CLIP_MIN_SCORE} \
  --group-margin-min {GROUP_MARGIN_MIN} \
  --human-photo-min {HUMAN_PHOTO_MIN} \
  {'--disable-cartoon-check' if DISABLE_CARTOON_CHECK else ''} \
  {'--disable-toy-check' if DISABLE_TOY_CHECK else ''} \
  --align-on {ALIGN_ON} \
  --copy-files
""".strip()
quality_cmd = " ".join(quality_cmd.split())
print(quality_cmd)
!{quality_cmd}


## 11. Check Stage 02 Output

Purpose:

- verify `metadata_balanced.csv` exists
- inspect balanced counts
- inspect before/after filtering summaries

In [ ]:
metadata_balanced_path = project_root / "data" / "metadata_balanced.csv"
assert metadata_balanced_path.exists(), f"Missing Stage 02 output: {metadata_balanced_path}"
metadata_balanced = pd.read_csv(metadata_balanced_path)
assert len(metadata_balanced) > 0, "metadata_balanced.csv is empty"
print(metadata_balanced.groupby(["group", "y_true"]).size())
print("total:", len(metadata_balanced))


In [ ]:
!cat /content/drive/MyDrive/project/results/fairness_tables/quality_clip_summary_before.csv
!cat /content/drive/MyDrive/project/results/fairness_tables/quality_clip_summary_after.csv


## 12. Stage 03 Run Detectors

Purpose:

- run your current detector set: `CNNDetection + LGrad + NPR`
- generate detector score CSVs and summaries

In [ ]:
!python /content/drive/MyDrive/project/scripts/03_run_detectors.py --project-root /content/drive/MyDrive/project --metadata-in /content/drive/MyDrive/project/data/metadata_balanced.csv --detector cnndetection --device cuda
!python /content/drive/MyDrive/project/scripts/03_run_detectors.py --project-root /content/drive/MyDrive/project --metadata-in /content/drive/MyDrive/project/data/metadata_balanced.csv --detector lgrad --device cuda
!python /content/drive/MyDrive/project/scripts/03_run_detectors.py --project-root /content/drive/MyDrive/project --metadata-in /content/drive/MyDrive/project/data/metadata_balanced.csv --detector npr --device cuda


In [ ]:
!cat /content/drive/MyDrive/project/results/detector_outputs/cnndetection_summary.txt
!cat /content/drive/MyDrive/project/results/detector_outputs/lgrad_summary.txt
!cat /content/drive/MyDrive/project/results/detector_outputs/npr_summary.txt


In [ ]:
!python /content/drive/MyDrive/project/scripts/03_run_detectors.py --project-root /content/drive/MyDrive/project --metadata-in /content/drive/MyDrive/project/data/metadata_balanced.csv --detector cnndetection,lgrad,npr --device cuda
!ls -lh /content/drive/MyDrive/project/results/detector_outputs


## 13. Stage 04 Fairness Evaluation

Purpose:

- run fairness evaluation for all three detectors
- generate fairness summaries and bootstrap intervals

In [ ]:
!python /content/drive/MyDrive/project/scripts/04_fairness_eval.py --detector-csv /content/drive/MyDrive/project/results/detector_outputs/cnndetection_scores.csv --output-dir /content/drive/MyDrive/project/results/fairness_tables/cnndetection --split test --bootstrap-iters 1000 --seed 42
!python /content/drive/MyDrive/project/scripts/04_fairness_eval.py --detector-csv /content/drive/MyDrive/project/results/detector_outputs/lgrad_scores.csv --output-dir /content/drive/MyDrive/project/results/fairness_tables/lgrad --split test --bootstrap-iters 1000 --seed 42
!python /content/drive/MyDrive/project/scripts/04_fairness_eval.py --detector-csv /content/drive/MyDrive/project/results/detector_outputs/npr_scores.csv --output-dir /content/drive/MyDrive/project/results/fairness_tables/npr --split test --bootstrap-iters 1000 --seed 42


In [ ]:
!find /content/drive/MyDrive/project/results/fairness_tables -maxdepth 2 -type f
!cat /content/drive/MyDrive/project/results/fairness_tables/cnndetection/fairness_summary.json
!cat /content/drive/MyDrive/project/results/fairness_tables/lgrad/fairness_summary.json
!cat /content/drive/MyDrive/project/results/fairness_tables/npr/fairness_summary.json


## 14. Stage 06 Consolidate Latest Main Results

Purpose:

- consolidate the current main detector set into one overview table

In [ ]:
!python /content/drive/MyDrive/project/scripts/06_consolidate_results.py --project-root /content/drive/MyDrive/project --metadata /content/drive/MyDrive/project/data/metadata_balanced.csv --detectors cnndetection,lgrad,npr --output-csv /content/drive/MyDrive/project/results/fairness_tables/latest_run_overview.csv --output-notes /content/drive/MyDrive/project/results/fairness_tables/latest_run_notes.md
!cat /content/drive/MyDrive/project/results/fairness_tables/latest_run_notes.md


In [ ]:
latest_overview = pd.read_csv('/content/drive/MyDrive/project/results/fairness_tables/latest_run_overview.csv')
latest_overview


## 15. Stage 05 Grad-CAM Attribution

Purpose:

- generate Grad-CAM attribution maps for your current detectors

Important:

- source code has been updated to support `npr`
- this step now aligns with your current detector set

In [ ]:
!python /content/drive/MyDrive/project/scripts/05_gradcam_analysis.py --project-root /content/drive/MyDrive/project --detector-csv /content/drive/MyDrive/project/results/detector_outputs/cnndetection_scores.csv --output-dir /content/drive/MyDrive/project/results/attribution/cnndetection --analyze-all --max-per-group 2 --device cuda
!python /content/drive/MyDrive/project/scripts/05_gradcam_analysis.py --project-root /content/drive/MyDrive/project --detector-csv /content/drive/MyDrive/project/results/detector_outputs/lgrad_scores.csv --output-dir /content/drive/MyDrive/project/results/attribution/lgrad --analyze-all --max-per-group 2 --device cuda
!python /content/drive/MyDrive/project/scripts/05_gradcam_analysis.py --project-root /content/drive/MyDrive/project --detector-csv /content/drive/MyDrive/project/results/detector_outputs/npr_scores.csv --output-dir /content/drive/MyDrive/project/results/attribution/npr --analyze-all --max-per-group 2 --device cuda
!find /content/drive/MyDrive/project/results/attribution -maxdepth 3 -type f


## 16. Stage 06 Patch Shuffling Structural Attribution

Purpose:

- test whether your detectors rely more on local texture or global structure
- default detector set is now aligned to `cnndetection,lgrad,npr`

In [ ]:
!python /content/drive/MyDrive/project/scripts/06_patch_shuffling_exp.py --project-root /content/drive/MyDrive/project --patch-scales 1,2,4,8,16,32 --detectors cnndetection,lgrad,npr --seed 42
!find /content/drive/MyDrive/project/results/structural_attribution -maxdepth 2 -type f


## 17. Stage 07 Physical Consistency Analysis

Purpose:

- extract second-order physical statistics from real and fake images

In [ ]:
!python /content/drive/MyDrive/project/scripts/07_physical_consistency.py --project-root /content/drive/MyDrive/project --max-samples 50
!ls -lh /content/drive/MyDrive/project/data/physical_consistency_results.csv


## 18. Stage 08 High-Pass Residual Generation

Purpose:

- generate high-pass residual versions of real and fake images
- this script itself is detector-agnostic
- detector support comes from the next step via `03_run_detectors.py`

In [ ]:
!python /content/drive/MyDrive/project/scripts/08_high_pass_innovation.py --project-root /content/drive/MyDrive/project --process-all
!find /content/drive/MyDrive/project/data/high_pass_residuals -maxdepth 3 -type d


## 19. Run Current Detectors on High-Pass Residuals

Purpose:

- evaluate your current detector set on `high_pass_residuals`
- source code path has been aligned with your current detectors
- outputs are written to `results/detector_outputs_highpass`


In [ ]:
!python /content/drive/MyDrive/project/scripts/03_run_detectors.py --project-root /content/drive/MyDrive/project --input-dir /content/drive/MyDrive/project/data/high_pass_residuals --detector cnndetection,lgrad,npr --output-dir /content/drive/MyDrive/project/results/detector_outputs_highpass --device cuda
!ls -lh /content/drive/MyDrive/project/results/detector_outputs_highpass


## 20. Stage 09 Master Report

Purpose:

- consolidate fairness, physical consistency, patch shuffling and high-pass residual results
- default detector set is now aligned to `cnndetection,lgrad,npr`

In [ ]:
!python /content/drive/MyDrive/project/scripts/09_master_report.py --project-root /content/drive/MyDrive/project --detectors cnndetection,lgrad,npr
!find /content/drive/MyDrive/project/results/master_report -maxdepth 2 -type f
!cat /content/drive/MyDrive/project/results/master_report/MASTER_REPORT.md


## Final Notes

This notebook now matches your actual requirements:

- removed the separate NPR download step
- real image flow is manual, not crawler-based
- Grad-CAM source code has been updated to support `npr`
- high-pass residual evaluation is aligned to your current detector set
- patch shuffling and master report defaults also use `cnndetection,lgrad,npr`
